In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'r_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.05
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: Rate
Augmented coefficient: r_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.604 0.    0.   ]
 [0.    0.839 0.   ]
 [0.    0.    0.039]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,17.293,0.002,0.022,0.0,100000,99238,0.992,0.0,0.992,0.993
1,Infl,27.050,0.000,0.026,0.0,100000,99993,1.000,0.0,1.000,1.000
2,Rate,15.579,0.004,0.022,0.0,100000,98085,0.981,0.0,0.980,0.982


In [8]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 100000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,2.401,0.589,0.000,0.007,0.001,100000,3468,0.035,0.001,0.034,0.036,3.0,200,4
1,cov_identity,16.530,298.647,0.000,0.006,0.290,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-1.096,-0.050,1.561,-0.712,0.434,0.007,0.005,0.0,0.000,0.003,0.001,0.0,100000,9754,0.098,0.001,0.096,0.099
1,OutGap,x,-0.434,-0.156,0.193,-2.236,0.104,0.029,0.001,0.0,0.000,0.003,0.001,0.0,100000,60396,0.604,0.002,0.601,0.607
2,OutGap,r,-0.667,-0.023,1.992,-0.321,0.494,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,5323,0.053,0.001,0.052,0.055
3,Infl,Pi,-0.907,-0.035,1.771,-0.497,0.461,0.006,0.006,0.0,0.000,0.003,0.001,0.0,100000,7950,0.080,0.001,0.078,0.081
4,Infl,x,0.078,0.025,0.222,0.355,0.478,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,6713,0.067,0.001,0.066,0.069
5,Infl,r,-0.527,-0.015,2.260,-0.215,0.488,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,5816,0.058,0.001,0.057,0.060
6,Rate,Pi,-0.201,-0.043,0.340,-0.606,0.449,0.007,0.001,0.0,0.000,0.003,0.001,0.0,100000,8979,0.090,0.001,0.088,0.092
7,Rate,x,0.023,0.039,0.043,0.548,0.457,0.006,0.000,0.0,0.000,0.003,0.001,0.0,100000,8342,0.083,0.001,0.082,0.085
8,Rate,r,-0.544,-0.087,0.431,-1.236,0.311,0.012,0.001,0.0,0.000,0.003,0.001,0.0,100000,22927,0.229,0.001,0.227,0.232


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-3.885,-0.310,0.841,-4.605,0.001,0.099,0.003,0.0,0.000,0.003,0.000,0.0,100000,99910,0.999,0.000,0.999,0.999
1,OutGap,x,-0.523,-0.341,0.101,-5.131,0.000,0.119,0.000,0.0,0.000,0.003,0.000,0.0,100000,99984,1.000,0.000,1.000,1.000
0,OutGap,r,1.490,0.055,1.884,0.779,0.454,0.006,0.004,0.0,0.001,0.002,0.001,0.0,100000,4968,0.050,0.001,0.048,0.051
5,Infl,Pi,-0.255,-0.017,1.003,-0.236,0.493,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,5493,0.055,0.001,0.054,0.056
4,Infl,x,0.003,0.004,0.122,0.050,0.504,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,4712,0.047,0.001,0.046,0.048
3,Infl,r,-0.578,-0.018,2.139,-0.260,0.490,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,5587,0.056,0.001,0.054,0.057
8,Rate,Pi,-0.032,-0.013,0.192,-0.185,0.503,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,4787,0.048,0.001,0.047,0.049
7,Rate,x,0.009,0.026,0.023,0.374,0.485,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,6021,0.060,0.001,0.059,0.062
6,Rate,r,-0.582,-0.098,0.408,-1.397,0.273,0.014,0.001,0.0,0.000,0.003,0.001,0.0,100000,28160,0.282,0.001,0.279,0.284


In [11]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.594019,-2.690120,-1.096101,-1.096101,-1.385439e-18,4.280401e-16,1.768752e-15,0.002982,0.002456,0.004728,0.004728,1.755848e-18,1.118400e-18,8.184537e-19
1,OutGap,x,0.032561,-0.467037,-0.434476,-0.434476,-9.968094e-20,6.420061e-17,1.768752e-15,0.000372,0.000345,0.000624,0.000624,2.788490e-19,1.911524e-19,8.184537e-19
2,OutGap,r,-0.269047,-0.398121,-0.667169,-0.667169,1.249366e-18,4.384983e-16,1.768752e-15,0.003850,0.003296,0.006169,0.006169,1.837048e-18,1.204962e-18,8.184537e-19
3,Infl,Pi,-0.037012,-0.870306,-0.907319,-0.907319,-1.553069e-17,4.347587e-16,1.768752e-15,0.001280,0.005571,0.005697,0.005697,1.814833e-18,1.185690e-18,8.184537e-19
4,Infl,x,0.003988,0.074183,0.078171,0.078171,-4.475352e-19,5.357103e-17,1.768752e-15,0.000161,0.000697,0.000713,0.000713,2.229277e-19,1.449136e-19,8.184537e-19
5,Infl,r,-0.017542,-0.509839,-0.527380,-0.527380,1.599339e-18,5.436590e-16,1.768752e-15,0.001640,0.007196,0.007344,0.007344,2.267807e-18,1.478950e-18,8.184537e-19
6,Rate,Pi,0.000961,-0.201859,-0.200899,-0.200899,2.004444e-21,1.586646e-16,1.768752e-15,0.000275,0.001039,0.001058,0.001058,6.370198e-19,3.924882e-19,8.184537e-19
7,Rate,x,-0.000203,0.023517,0.023314,0.023314,-4.968364e-20,1.983404e-17,1.768752e-15,0.000035,0.000133,0.000135,0.000135,7.965825e-20,4.910733e-20,8.184537e-19
8,Rate,r,-0.007820,-0.536631,-0.544451,-0.544451,-4.691739e-19,2.135095e-16,1.768752e-15,0.000351,0.001415,0.001423,0.001423,8.643860e-19,5.397202e-19,8.184537e-19


In [12]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.874825e+00,-5.760226,-3.885401,-3.885401,-1.808553e-18,5.820089e-16,1.768752e-15,0.001670,0.001209,0.002581,0.002581,2.466764e-18,1.642431e-18,8.184537e-19
1,OutGap,x,1.992981e-01,-0.721876,-0.522578,-0.522578,-1.269818e-19,7.312220e-17,1.768752e-15,0.000206,0.000191,0.000336,0.000336,3.134696e-19,2.116465e-19,8.184537e-19
2,OutGap,r,-7.644014e-01,2.254318,1.489916,1.489916,-1.293142e-18,4.590171e-16,1.768752e-15,0.004424,0.003534,0.004451,0.004451,1.911634e-18,1.243934e-18,8.184537e-19
3,Infl,Pi,-5.386468e-03,-0.249585,-0.254971,-0.254971,-1.883887e-17,2.417984e-16,1.768752e-15,0.000726,0.003101,0.003170,0.003170,1.000405e-18,6.478345e-19,8.184537e-19
4,Infl,x,2.817738e-04,0.003138,0.003420,0.003420,-2.069743e-18,2.941852e-17,1.768752e-15,0.000088,0.000375,0.000383,0.000383,1.211718e-19,7.791581e-20,8.184537e-19
5,Infl,r,-1.905331e-02,-0.558790,-0.577843,-0.577843,6.783859e-18,5.151270e-16,1.768752e-15,0.001547,0.006669,0.006791,0.006791,2.143011e-18,1.392613e-18,8.184537e-19
6,Rate,Pi,-2.424092e-05,-0.031757,-0.031782,-0.031782,2.362974e-19,8.890556e-17,1.768752e-15,0.000156,0.000578,0.000591,0.000591,3.552953e-19,2.172384e-19,8.184537e-19
7,Rate,x,4.753842e-07,0.009279,0.009279,0.009279,2.713113e-21,1.090997e-17,1.768752e-15,0.000019,0.000072,0.000073,0.000073,4.368942e-20,2.680445e-20,8.184537e-19
8,Rate,r,-5.348574e-03,-0.576734,-0.582082,-0.582082,-3.146450e-20,2.052465e-16,1.768752e-15,0.000332,0.001359,0.001368,0.001368,8.337595e-19,5.233445e-19,8.184537e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [13]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

### Marginal LR Test Conditional on $\theta_0$

In [14]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,-0.417,-2510.468,-2497.238,26.46,0.015,0.0,0.627,0.617,0.053,0.0,100000,95679,0.957,0.001,0.956,0.958


In [15]:
res_mle

OptimizationResult(kind='mle', x=array([-0.31774282]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.0), 'r_coef': np.float64(-0.3177428221896167)}, success=True, message='CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', fun=np.float64(2151.523881234458), loglik=np.float64(-2151.523881234458), logprior=np.float64(0.0), logpost=np.float64(-2151.523881234458), nfev=12, nit=4, raw=  message: CONVERGENCE:

## Serial Autocorrelation Tests for the Augmented Model

In [16]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,17.304,0.002,0.022,0.0,100000,99242,0.992,0.000,0.992,0.993
1,Infl,27.048,0.000,0.026,0.0,100000,99993,1.000,0.000,1.000,1.000
2,Rate,13.181,0.010,0.020,0.0,100000,95139,0.951,0.001,0.950,0.953


In [17]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.122,2.401,0.589,0.000,0.007,0.001,100000,3468,0.035,0.001,0.034,0.036,3.0,200,4
1,cov_identity,16.530,298.647,0.000,0.006,0.290,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.126,2.416,0.587,0.000,0.007,0.001,100000,3505,0.035,0.001,0.034,0.036,3.0,200,4
1,cov_identity,16.252,289.137,0.000,0.006,0.271,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
